# De Novo Molecular Generation with Graph Neural Networks on MOSES (ZINC-derived)

**Author:** Babak Mamnoon  
**Environment:** Google Colab · PyTorch · PyTorch Geometric · RDKit · MOSES dataset  

This notebook implements an industry-grade de novo molecular generation pipeline using:

- **Dataset:** MOSES (refined from ZINC Clean Leads; ~1.9M molecules)   
- **Representation:** RDKit molecular graphs (atoms/bonds) + SMILES sequences  
- **Model:** Graph-based autoencoder  
  - **Encoder:** GNN (GCN-based) → continuous latent vector  
  - **Decoder:** GRU-based SMILES generator conditioned on latent  
- **Task:** Learn a latent space of drug-like molecules and generate novel compounds  
- **Evaluation:** Validity, uniqueness, novelty, property distributions (MW, logP, QED), latent visualization, training curves  

This design follows current practices in molecular generative modeling and GNN-based representation learning, inspired by MOSES, JT-VAE, and related literature. 


## Section 1. Environment Setup (Colab-compatible)

In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# Core chemistry / ML stack
install("rdkit-pypi")
install("torch")
install("torch-geometric")
install("pandas")
install("numpy")
install("scikit-learn")
install("matplotlib")
install("seaborn")
install("tqdm")

print("✅ All packages installed successfully.")

## Section 2. Imports & Global Configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os, math, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# RDKit
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdMolDescriptors, QED
RDLogger.DisableLog('rdApp.*')

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F

# PyTorch Geometric
from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool, global_max_pool

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥  Device : {DEVICE}")
print(f"📦 PyTorch : {torch.__version__}")

## Section 3. Dataset: MOSES (ZINC-derived)

We use the **MOSES** benchmark dataset, which is a curated subset of ZINC Clean Leads:

- ~1.9M molecules after medicinal chemistry and PAINS filtering  
- MW: 250–350 Da, rotatable bonds ≤ 7, XlogP ≤ 3.5  
- Only common drug-like atoms (C, N, O, S, F, Cl, Br, H)   

For this notebook, we will:

- Download the **train split** (`train.csv`) from the official MOSES GitHub  
- Use a manageable subset (e.g., 100k molecules) for Colab runtime  
- Represent molecules as both **SMILES strings** and **RDKit graphs**

In [ ]:
"""
Section 3.1 — Download MOSES train split (SMILES)
"""

MOSES_TRAIN_URL = "https://media.githubusercontent.com/media/molecularsets/moses/master/data/train.csv?raw=1"

print("⬇️  Downloading MOSES train split …")
df_moses = pd.read_csv(MOSES_TRAIN_URL)

print(f"Columns: {df_moses.columns.tolist()}")
print(f"Rows: {len(df_moses):,}")

# MOSES train.csv already has a column named 'SMILES'
if "SMILES" not in df_moses.columns:
    raise ValueError("The downloaded file does not contain a SMILES column.")

# Subsample for Colab
MAX_MOLECULES = 100_000
df_moses = df_moses.sample(n=min(MAX_MOLECULES, len(df_moses)), random_state=42).reset_index(drop=True)

print(f"Using subset: {len(df_moses):,} molecules")
df_moses.head()

## Section 4. Exploratory Data Analysis (EDA) on MOSES subset

In [ ]:
def mol_props(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    num_atoms = mol.GetNumAtoms()
    num_rings = rdMolDescriptors.CalcNumRings(mol)
    qed = QED.qed(mol)
    return mw, logp, num_atoms, num_rings, qed

print("Computing molecular properties for EDA …")
props = df_moses["SMILES"].apply(
    lambda s: pd.Series(mol_props(s), index=["MW","logP","NumAtoms","NumRings","QED"])
)
df_eda = pd.concat([df_moses, props], axis=1).dropna()
print(f"Valid molecules for EDA: {len(df_eda):,}")

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("MOSES Subset — Property Distributions", fontsize=16, fontweight="bold")

axes = axes.flatten()
for ax, col, label in zip(
    axes[:5],
    ["MW","logP","NumAtoms","NumRings","QED"],
    ["Molecular Weight (Da)","logP","Atom Count","# Rings","QED"]
):
    ax.hist(df_eda[col], bins=50, color="steelblue", alpha=0.8, edgecolor="none")
    ax.set_title(label)
    ax.grid(alpha=0.3)

axes[5].axis("off")

plt.tight_layout()
plt.show()

## Section 5. Molecular Graph Featurization (RDKit → PyG Data)

In [ ]:
# Atom features (similar to DTI example, tuned for MOSES)
def atom_features(atom):
    allowable_atoms = ['C','N','O','S','F','P','Cl','Br','I','other']
    def one_hot(val, choices):
        return [int(val == c) for c in choices]
    return (
        one_hot(atom.GetSymbol(), allowable_atoms) +
        one_hot(str(atom.GetHybridization()),
                ['SP','SP2','SP3','SP3D','SP3D2','other']) +
        [int(atom.GetIsAromatic())] +
        [atom.GetFormalCharge()] +
        [atom.GetTotalNumHs()] +
        [int(atom.IsInRing())]
    )

# Bond features
def bond_features(bond):
    bt = bond.GetBondType()
    return [
        int(bt == Chem.rdchem.BondType.SINGLE),
        int(bt == Chem.rdchem.BondType.DOUBLE),
        int(bt == Chem.rdchem.BondType.TRIPLE),
        int(bt == Chem.rdchem.BondType.AROMATIC),
        int(bond.GetIsConjugated()),
        int(bond.IsInRing()),
    ]

def smiles_to_graph(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    edge_idx, edge_attr = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bf = bond_features(bond)
        edge_idx += [[i,j],[j,i]]
        edge_attr += [bf, bf]
    if not edge_idx:
        return None
    edge_index = torch.tensor(edge_idx, dtype=torch.long).t().contiguous()
    edge_attr  = torch.tensor(edge_attr, dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

# Infer feature dims
_sm = Chem.MolFromSmiles("CCO")
NUM_ATOM_FEAT = len(atom_features(_sm.GetAtomWithIdx(0)))
NUM_BOND_FEAT = len(bond_features(list(_sm.GetBonds())[0]))
print(f"Atom feature dim : {NUM_ATOM_FEAT}")
print(f"Bond feature dim : {NUM_BOND_FEAT}")

## Section 6. SMILES Tokenization for Decoder

In [ ]:
# Simple character-level SMILES vocabulary
SMILES_CHARS = sorted(list(set("".join(df_moses["SMILES"].tolist()))))
SMILES_CHARS = SMILES_CHARS + ["<pad>","<bos>","<eos>"]
char2idx = {c: i for i, c in enumerate(SMILES_CHARS)}
idx2char = {i: c for c, i in char2idx.items()}
VOCAB_SIZE = len(SMILES_CHARS)
MAX_SMILES_LEN = 120  # cap for Colab

def encode_smiles(smi, max_len=MAX_SMILES_LEN):
    smi = smi[:max_len-2]  # reserve for BOS/EOS
    tokens = ["<bos>"] + list(smi) + ["<eos>"]
    ids = [char2idx[t] for t in tokens]
    if len(ids) < max_len:
        ids += [char2idx["<pad>"]] * (max_len - len(ids))
    else:
        ids = ids[:max_len]
    return torch.tensor(ids, dtype=torch.long)

print(f"SMILES vocab size: {VOCAB_SIZE}")
print(f"Max SMILES length: {MAX_SMILES_LEN}")

## Section 7. Dataset Class: Graph + SMILES

In [ ]:
from torch.utils.data import Dataset as TorchDataset

class GraphSmilesDataset(TorchDataset):
    """
    Each sample: (graph, smiles_ids, original_smiles)
    """
    def __init__(self, df):
        self.records = []
        skipped = 0
        for smi in tqdm(df["SMILES"], desc="Building GraphSmilesDataset"):
            graph = smiles_to_graph(smi)
            if graph is None:
                skipped += 1
                continue
            smiles_ids = encode_smiles(smi)
            self.records.append((graph, smiles_ids, smi))
        if skipped:
            print(f"Skipped {skipped} invalid SMILES")

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        return self.records[idx]

def collate_batch(batch):
    graphs, smiles_ids, smiles_str = zip(*batch)
    return (
        Batch.from_data_list(graphs),
        torch.stack(smiles_ids),
        list(smiles_str),
    )

dataset = GraphSmilesDataset(df_moses)
print(f"Dataset size: {len(dataset):,}")

BATCH_SIZE = 128
train_loader = torch.utils.data.DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch
)
print(f"Train batches: {len(train_loader)}")

## Section 8. Model Architecture: GNN Encoder + GRU Decoder

We implement a **graph-based autoencoder**:

1. **Graph Encoder (DrugEncoderGNN)**  
   - 3-layer GCN with residual connections and global mean+max pooling  
   - Outputs a fixed-size latent vector `z` (e.g., 256-dim)

2. **SMILES Decoder (SmilesDecoderGRU)**  
   - GRU-based sequence decoder conditioned on `z`  
   - Teacher forcing during training  
   - Generates SMILES tokens autoregressively at inference

3. **Training Objective**  
   - Cross-entropy loss over SMILES tokens  
   - The encoder learns a latent space of drug-like molecules; decoder reconstructs SMILES  
   - At generation time, we sample `z` from a Gaussian prior fitted to training latents and decode new SMILES

In [ ]:
"""
Section 8.1 — GNN Encoder
"""

class ResGCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.1):
        super().__init__()
        self.conv = GCNConv(in_ch, out_ch)
        self.bn   = nn.BatchNorm1d(out_ch)
        self.drop = nn.Dropout(dropout)
        self.proj = nn.Linear(in_ch, out_ch, bias=False) if in_ch != out_ch else nn.Identity()

    def forward(self, x, edge_index):
        out = F.relu(self.bn(self.conv(x, edge_index)))
        out = self.drop(out)
        return out + self.proj(x)

class DrugEncoderGNN(nn.Module):
    def __init__(self, num_atom_feat, hidden=128, latent_dim=256, dropout=0.15):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(num_atom_feat, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU()
        )
        self.gcn1 = ResGCNBlock(hidden, hidden, dropout)
        self.gcn2 = ResGCNBlock(hidden, hidden, dropout)
        self.gcn3 = ResGCNBlock(hidden, hidden, dropout)
        self.fc   = nn.Sequential(
            nn.Linear(2*hidden, latent_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.out_dim = latent_dim

    def forward(self, batch_graph):
        x = self.proj(batch_graph.x.float())
        x = self.gcn1(x, batch_graph.edge_index)
        x = self.gcn2(x, batch_graph.edge_index)
        x = self.gcn3(x, batch_graph.edge_index)
        h_mean = global_mean_pool(x, batch_graph.batch)
        h_max  = global_max_pool(x, batch_graph.batch)
        h = torch.cat([h_mean, h_max], dim=1)
        z = self.fc(h)
        return z

"""
Section 8.2 — GRU SMILES Decoder
"""

class SmilesDecoderGRU(nn.Module):
    def __init__(self, vocab_size, latent_dim=256, embed_dim=128, hidden_dim=256, num_layers=2, dropout=0.2):
        super().__init__()
        self.vocab_size = vocab_size
        self.latent_dim = latent_dim
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=char2idx["<pad>"])
        self.latent_to_hidden = nn.Linear(latent_dim, hidden_dim)
        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, z, smiles_ids):
        """
        z: (B, latent_dim)
        smiles_ids: (B, L) — teacher forcing
        """
        B, L = smiles_ids.size()
        emb = self.embed(smiles_ids)  # (B, L, E)
        h0 = self.latent_to_hidden(z).unsqueeze(0).repeat(self.gru.num_layers, 1, 1)
        out, _ = self.gru(emb, h0)
        logits = self.fc_out(out)  # (B, L, V)
        return logits

    def sample(self, z, max_len=MAX_SMILES_LEN, greedy=True):
        """
        Autoregressive generation from latent z.
        """
        B = z.size(0)
        h = self.latent_to_hidden(z).unsqueeze(0).repeat(self.gru.num_layers, 1, 1)
        inputs = torch.full((B, 1), char2idx["<bos>"], dtype=torch.long, device=z.device)
        generated = []

        for _ in range(max_len):
            emb = self.embed(inputs)  # (B, 1, E)
            out, h = self.gru(emb, h)
            logits = self.fc_out(out[:, -1, :])  # (B, V)
            probs = F.softmax(logits, dim=-1)
            if greedy:
                next_ids = torch.argmax(probs, dim=-1, keepdim=True)
            else:
                next_ids = torch.multinomial(probs, num_samples=1)
            generated.append(next_ids.squeeze(1).cpu().numpy())
            inputs = next_ids
        # Convert to SMILES strings
        smiles_list = []
        for b in range(B):
            tokens = []
            for t_id in [gen[b] for gen in generated]:
                ch = idx2char[int(t_id)]
                if ch == "<eos>":
                    break
                if ch not in ["<bos>","<pad>"]:
                    tokens.append(ch)
            smiles_list.append("".join(tokens))
        return smiles_list

"""
Section 8.3 — Full Autoencoder Model
"""

class GraphSmilesAutoencoder(nn.Module):
    def __init__(self, num_atom_feat, vocab_size, hidden=128, latent_dim=256, dropout=0.2):
        super().__init__()
        self.encoder = DrugEncoderGNN(num_atom_feat, hidden=hidden, latent_dim=latent_dim, dropout=dropout)
        self.decoder = SmilesDecoderGRU(vocab_size, latent_dim=latent_dim, embed_dim=128,
                                        hidden_dim=256, num_layers=2, dropout=dropout)

    def forward(self, batch_graph, smiles_ids):
        z = self.encoder(batch_graph)
        logits = self.decoder(z, smiles_ids)
        return logits, z

model = GraphSmilesAutoencoder(NUM_ATOM_FEAT, VOCAB_SIZE, hidden=128, latent_dim=256, dropout=0.2).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"🔢 Trainable parameters: {n_params:,}")

## Section 9. Training Pipeline

In [ ]:
EPOCHS       = 20
LR           = 1e-3
WEIGHT_DECAY = 1e-5

criterion = nn.CrossEntropyLoss(ignore_index=char2idx["<pad>"])
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    all_latents = []

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch_graph, smiles_ids, _ in loader:
            batch_graph = batch_graph.to(DEVICE)
            smiles_ids  = smiles_ids.to(DEVICE)

            if train:
                optimizer.zero_grad()

            logits, z = model(batch_graph, smiles_ids)
            # Shift targets by one (predict next token)
            targets = smiles_ids[:, 1:]  # (B, L-1)
            logits  = logits[:, :-1, :]  # (B, L-1, V)
            loss = criterion(logits.reshape(-1, VOCAB_SIZE), targets.reshape(-1))

            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item() * smiles_ids.size(0)
            all_latents.append(z.detach().cpu())

    avg_loss = total_loss / len(loader.dataset)
    latents = torch.cat(all_latents, dim=0)
    return avg_loss, latents

train_losses = []
print("🚀 Starting training …\n")
for epoch in range(1, EPOCHS+1):
    tr_loss, latents = run_epoch(train_loader, train=True)
    train_losses.append(tr_loss)
    print(f"Epoch {epoch:>3} | Train Loss: {tr_loss:.4f}")

print("\n✅ Training complete.")

## Section 10. Learning Curves

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 5))
ax.plot(range(1, len(train_losses)+1), train_losses, marker="o", color="steelblue")
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-Entropy Loss")
ax.set_title("Training Loss — GraphSmiles Autoencoder")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Section 11. Latent Space Visualization (t-SNE)

In [ ]:
from sklearn.manifold import TSNE

# Collect latents from a single pass
_, latents = run_epoch(train_loader, train=False)
print(f"Latent tensor shape: {latents.shape}")

# Subsample for t-SNE
N_TSNE = 5000
idx = np.random.choice(latents.shape[0], size=min(N_TSNE, latents.shape[0]), replace=False)
lat_sample = latents[idx].numpy()

tsne = TSNE(n_components=2, random_state=SEED, perplexity=30)
lat_2d = tsne.fit_transform(lat_sample)

plt.figure(figsize=(7, 6))
plt.scatter(lat_2d[:,0], lat_2d[:,1], s=5, alpha=0.6, c="steelblue")
plt.title("Latent Space (t-SNE) — GraphSmiles Autoencoder")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Section 12. De Novo Generation & Evaluation

In [ ]:
def generate_molecules(n_samples=1000, greedy=True):
    model.eval()
    with torch.no_grad():
        # Sample z from Gaussian prior fitted to encoder latents
        _, latents = run_epoch(train_loader, train=False)
        mu = latents.mean(dim=0)
        cov = latents.std(dim=0)
        z = torch.randn(n_samples, latents.shape[1]) * cov + mu
        z = z.to(DEVICE)
        smiles_gen = model.decoder.sample(z, max_len=MAX_SMILES_LEN, greedy=greedy)
    return smiles_gen

print("Generating molecules …")
gen_smiles = generate_molecules(n_samples=2000, greedy=False)
print(f"Generated {len(gen_smiles)} SMILES")

def validity(smiles_list):
    valid = []
    for s in smiles_list:
        mol = Chem.MolFromSmiles(s)
        if mol is not None:
            valid.append(s)
    return valid

valid_smiles = validity(gen_smiles)
valid_ratio = len(valid_smiles) / len(gen_smiles)
print(f"Validity: {valid_ratio*100:.2f}% ({len(valid_smiles)}/{len(gen_smiles)})")

unique_smiles = list(set(valid_smiles))
unique_ratio = len(unique_smiles) / len(valid_smiles) if valid_smiles else 0.0
print(f"Uniqueness: {unique_ratio*100:.2f}% ({len(unique_smiles)}/{len(valid_smiles)})")

# Novelty: fraction not in training set
train_set = set(df_moses["SMILES"].tolist())
novel_smiles = [s for s in unique_smiles if s not in train_set]
novel_ratio = len(novel_smiles) / len(unique_smiles) if unique_smiles else 0.0
print(f"Novelty: {novel_ratio*100:.2f}% ({len(novel_smiles)}/{len(unique_smiles)})")

## Section 13. Visualization of Generated Molecules (Sample of 50)

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw
from PIL import Image
import io

# Use the already computed `valid_smiles` from Section 12
N_SHOW = 50
sample_smiles = valid_smiles[:N_SHOW] if len(valid_smiles) >= N_SHOW else valid_smiles

mols = []
for s in sample_smiles:
    mol = Chem.MolFromSmiles(s)
    if mol is not None:
        mols.append(mol)

print(f"Showing {len(mols)} generated molecules (structures)")

# Generate RDKit grid image (returns a PIL Image-like object)
img = Draw.MolsToGridImage(
    mols,
    molsPerRow=10,
    subImgSize=(200, 200),
    legends=[f"{i+1}" for i in range(len(mols))],
    useSVG=False
)

# Display inline in Colab
display(img)

# Convert to real PIL Image for saving
if not isinstance(img, Image.Image):
    img = Image.open(io.BytesIO(img.data))

# Save to file
img.save("generated_molecules_sample.png")
print("Saved grid image → generated_molecules_sample.png")

## Section 14. Save Generated Molecules to CSV

In [ ]:
# Use the lists already computed in Section 12:
# gen_smiles      → all generated SMILES (including invalid)
# valid_smiles    → RDKit-validated SMILES
# unique_smiles   → unique valid SMILES
# novel_smiles    → unique + not in training set

df_generated = pd.DataFrame({
    "generated_smiles": gen_smiles,
    "is_valid": [Chem.MolFromSmiles(s) is not None for s in gen_smiles]
})

# Add unique + novel sets as separate CSVs (optional but industry‑standard)
df_valid = pd.DataFrame({"valid_smiles": valid_smiles})
df_unique = pd.DataFrame({"unique_smiles": unique_smiles})
df_novel = pd.DataFrame({"novel_smiles": novel_smiles})

# Save files
df_generated.to_csv("generated_all.csv", index=False)
df_valid.to_csv("generated_valid.csv", index=False)
df_unique.to_csv("generated_unique.csv", index=False)
df_novel.to_csv("generated_novel.csv", index=False)

print("Saved:")
print(" - generated_all.csv")
print(" - generated_valid.csv")
print(" - generated_unique.csv")
print(" - generated_novel.csv")

## Section 15. Property Distributions: Train vs Generated

In [ ]:
def compute_props(smiles_list):
    rows = []
    for s in smiles_list:
        mol = Chem.MolFromSmiles(s)
        if mol is None:
            continue
        mw = Descriptors.MolWt(mol)
        logp = Descriptors.MolLogP(mol)
        qed = QED.qed(mol)
        rows.append((mw, logp, qed))
    return pd.DataFrame(rows, columns=["MW","logP","QED"])

train_props = compute_props(df_moses["SMILES"].tolist())
gen_props   = compute_props(valid_smiles)

print(f"Train props: {len(train_props):,} | Generated props: {len(gen_props):,}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Property Distributions — Train vs Generated", fontsize=14, fontweight="bold")

for ax, col, label in zip(
    axes,
    ["MW","logP","QED"],
    ["Molecular Weight (Da)","logP","QED"]
):
    sns.kdeplot(train_props[col], ax=ax, label="Train", color="steelblue")
    sns.kdeplot(gen_props[col], ax=ax, label="Generated", color="darkorange")
    ax.set_title(label)
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


## Section 16. Summary & References

In [ ]:
We implemented a **GNN-based de novo molecular generation pipeline** on the **MOSES** dataset:

- **Encoder:** GCN-based graph encoder producing a continuous latent space of drug-like molecules  
- **Decoder:** GRU-based SMILES generator conditioned on latent vectors  
- **Metrics:**  
  - Training loss curves  
  - Latent space visualization (t-SNE)  
  - Validity, uniqueness, novelty of generated molecules  
  - Property distribution alignment (MW, logP, QED) between train and generated sets  

This notebook can be extended with:

- MOSES official metrics (Fréchet ChemNet Distance, internal diversity, etc.)   
- Conditional generation (e.g., optimizing logP, QED, or other objectives)  
- More advanced decoders (graph-based decoders, JT-VAE-style architectures)

**Key references:**

- Polykovskiy et al., *Molecular Sets (MOSES): A Benchmarking Platform for Molecular Generation Models*, Frontiers in Pharmacology, 2020.   
- Jin et al., *Junction Tree Variational Autoencoder for Molecular Graph Generation*, ICML 2018.   
- RDKit: Open-source cheminformatics toolkit.
